In [ ]:
#%pip install langchain-core langchain-google-genai requests

In [1]:
import os
os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6L78QbE7ni0nOCPLhtlQqNbqG7nFisH9gj71YoMs64Txg"
os.environ["GROQ_API_KEY"] = "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"

第一種：使用wttr.in

In [ ]:
import requests
from langchain_core.tools import tool

@tool
def get_current_weather(location: str) -> str:
    """
    查詢指定地點的當前天氣狀況。
    
    Args:
        location (str): 城市或地點的名稱 (例如: "台北市", "Tokyo")
    
    Returns:
        str: 包含天氣描述與溫度的文字結果。
    """
    # 使用 free weather API (wttr.in) 作為查詢(需要可換成其他天氣 API)

    try:
        # 使用 format=3 會回傳簡短的文字結果，例如 "Taipei: 🌦️ +27°C"
        url = f"https://wttr.in/{location}?format=3"
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            return response.text.strip()
        else:
            return f"無法取得 {location} 的天氣資訊。"
    except Exception as e:
        return f"查詢天氣時發生錯誤: {str(e)}"

第二種：使用weather.api

In [ ]:
# weather_API
import requests
from langchain_core.tools import tool


#最好還是建立在建立環境變數
WEATHER_API_KEY = "188841f1b8d646008a371041262906"

@tool
def get_current_weather(location: str) -> str:
    """查詢指定城市或地區的目前天氣狀態。
    
    Args:
        location: 城市或地區名稱，支援中文或英文（例如：'台北'、'Taichung'、'東京'）。
    """
    url = "http://api.weatherapi.com/v1/current.json"
    
    # 設定請求參數
    params = {
        "key": WEATHER_API_KEY,
        "q": location,
        "aqi": "yes",       # 空氣品質資料
        "lang": "zh_tw"     # 讓天氣狀態敘述直接回傳繁體中文
    }
    
    try:
        response = requests.get(url, params=params)
        
        # 檢查回傳狀態碼，若非 200 則表示查詢失敗
        if response.status_code != 200:
            return f"找不到關於 '{location}' 的天氣資訊，請確認地名是否正確。"
            
        data = response.json()
        
        # 解析資料
        city = data["location"]["name"]           # 城市名
        region = data["location"]["region"]       # 縣市/區域
        temp_c = data["current"]["temp_c"]        # 攝氏溫度
        condition = data["current"]["condition"]["text"]  # 天氣狀態（如：晴、多雲）
        humidity = data["current"]["humidity"]    # 濕度
        pm25 = data["current"]["air_quality"]["pm2_5"] # PM2.5 指數
        
        # 回傳給 Agent
        result = (
            f"{region}{city}目前的天氣狀況為：{condition}。\n"
            f"現在氣溫 {temp_c}°C，相對濕度 {humidity}%。\n"
            f"當地 PM2.5 空氣品質指數為：{pm25:.1f}。"
        )
        return result
        
    except Exception as e:
        return f"與天氣服務連線時發生錯誤：{str(e)}"

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

# #初始化 Gemini 模型
# llm = ChatGoogleGenerativeAI(
#     model="gemini-1.5-flash",
#     temperature=0
# )

llm = ChatGroq(
    model="llama-3.3-70b-versatile", 
    temperature=0
)

In [4]:
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 建立 Agent(create_agent/create_tool_calling_agent)
agent = create_agent(
    model = llm,
    tools = [get_current_weather],
    system_prompt = "你是一個貼心的生活助手。如果使用者詢問天氣，請善用天氣查詢工具來回答。請用繁體中文回覆。"
    )

In [11]:
# 1. 設定user輸入
#user_input = "請問三重現在的天氣怎麼樣？"
user_input = input("請輸入要查詢的地點或問題：")


# 2. 以訊息格式傳給 agent
# 有需要config參數?
response = agent.invoke({
    "messages": [("user", user_input)]
})

# 3. 印出回覆
print(response["messages"][-1].content)

台北市目前的天氣狀況為：局部多雲。
現在氣溫 32.3°C，相對濕度 71%。
當地 PM2.5 空氣品質指數為：31.0。
